# 4. Evaluate

Avalia `eval_datasets` do experimento e salva `test_metrics.json` no run.

Pesos: run de treino do `experiment_id` (ou `cfg['weights']` quando não houver run).

## Preparação

In [1]:
import sys
sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from src import config, datasets, experiments, eval as eval_mod

## Configuração

In [2]:
# Experimento em `experiments/` (mesmo nome usado no treino)
EXPERIMENT = "e0-baseline"

N_WANDB_SAMPLE_PREDICTIONS = 10   # 0 desativa
WEIGHTS_OVERRIDE           = ""   # vazio = usa pesos do run

In [3]:
# Opcional: config completa inline
CONFIG = None

# CONFIG = {
#     "experiment_id": "e2-all",
#     "strategy": "direct",
#     "weights": "yolo11m.pt",
#     "dataset_stage": "interim",
#     "train_datasets": ["passenger-detection-bus", "inside-bus-view", "passenger-deakin"],
#     "eval_datasets":  ["passenger-detection-bus", "inside-bus-view", "passenger-deakin"],
#     "train_config": {
#         "epochs": 100, "patience": 20, "batch": 16, "imgsz": 640,
#         "workers": 4, "amp": True, "seed": 42, "deterministic": True,
#     },
# }

## Configuração resolvida

In [4]:
cfg = experiments.resolve(EXPERIMENT, CONFIG)

print("Experimentos disponíveis:", experiments.available())
print("Datasets disponíveis:    ", datasets.available(stage=cfg.get("dataset_stage", "interim")))
cfg

Experimentos disponíveis: ['e0-baseline', 'e1-dkn', 'e1-ibv', 'e1-pdb', 'e2-all', 'e2-all-aug-bad', 'e2-all-aug-bus', 'e2-all-aug-off', 'e2-all-proc', 'e2-ibv-dkn', 'e2-pdb-dkn', 'e2-pdb-ibv', 'e3-aug-bus', 'e3-crowd-all', 'e3-proc']
Datasets disponíveis:     ['crowdhuman', 'inside-bus-view', 'passenger-deakin', 'passenger-detection-bus']


{'experiment_id': 'e0-baseline',
 'strategy': 'baseline',
 'weights': 'yolo11m.pt',
 'dataset_stage': 'interim',
 'eval_datasets': ['passenger-detection-bus',
  'inside-bus-view',
  'passenger-deakin']}

## Avaliação

In [ ]:
metrics = eval_mod.run_experiment(
    cfg,
    n_wandb_samples=N_WANDB_SAMPLE_PREDICTIONS,
    weights_override=WEIGHTS_OVERRIDE,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lospe\_netrc.


eval:     nenhum run de treino — avaliando pesos base 'yolo11m.pt'
eval:     pesos -> yolo11m.pt


wandb: Currently logged in as: cristianmaza (IA901-2026S1-bus-passenger-count) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb:    run 'eval_e0-baseline_eval' -> https://wandb.ai/IA901-2026S1-bus-passenger-count/bus-passenger-count/runs/p4n4kuvc
Ultralytics 8.4.50  Python-3.12.5 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24576MiB)
YOLO11m summary (fused): 125 layers, 20,091,712 parameters, 0 gradients, 68.0 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 519.0110.0 MB/s, size: 217.7 KB)
val: Scanning C:\Users\lospe\Documents\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\interim\passenger-detection-bus\test\labels... 17 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 17/17 772.1it/s 0.0s
val: New cache created: C:\Users\lospe\Documents\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\interim\passenger-detection-bus\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3it/s 1.5s4.4s
                   all         17        213      0.112       0.23     0.0635     0.0209
                

## Resumo

In [ ]:
hdr = f"{'dataset':28s} {'mAP50':>8s} {'mAP50-95':>10s} {'prec':>8s} {'recall':>8s} {'F1':>8s} {'cMAE':>8s} {'cRMSE':>8s}"
print(hdr)
for ds, m in metrics.items():
    print(
        f"{ds:28s}"
        f" {m.get('metrics/mAP50(B)', float('nan')):8.4f}"
        f" {m.get('metrics/mAP50-95(B)', float('nan')):10.4f}"
        f" {m.get('metrics/precision(B)', float('nan')):8.4f}"
        f" {m.get('metrics/recall(B)', float('nan')):8.4f}"
        f" {m.get('F1', float('nan')):8.4f}"
        f" {m.get('count_mae', float('nan')):8.4f}"
        f" {m.get('count_rmse', float('nan')):8.4f}"
    )